In [1]:
import pyspark

In [19]:
from pyspark.sql import functions as F

In [2]:
BUCKET = "hw-bucket-sishabalova"

In [3]:
spark.version

'3.0.3'

In [17]:
s3_path = f"s3a://{BUCKET}/2019-08-22.txt"

In [18]:
raw_data = spark.read.text(s3_path)
raw_data.show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                |
+-------------------------------------------------------------------------------------------------------------------------------------+
|# tranaction_id | tx_datetime | customer_id | terminal_id | tx_amount | tx_time_seconds | tx_time_days | tx_fraud | tx_fraud_scenario|
|0,2019-08-22 06:51:03,0,711,70.91,24663,0,0,0                                                                                        |
|1,2019-08-22 05:10:37,0,0,90.55,18637,0,0,0                                                                                          |
|2,2019-08-22 19:05:33,0,753,35.38,68733,0,0,0                                                                                        |
|3,2019-08-22 07:21:33,0,0,80.41,26493,0,0,0    

In [20]:
raw_data_clean = raw_data.filter(~F.col("value").startswith("#"))

In [21]:
split_col = F.split(F.col("value"), ",")

In [22]:
df = raw_data_clean.select(
    split_col.getItem(0).cast("long").alias("tranaction_id"),
    split_col.getItem(1).alias("tx_datetime"),
    split_col.getItem(2).cast("long").alias("customer_id"),
    split_col.getItem(3).cast("long").alias("terminal_id"),
    split_col.getItem(4).cast("double").alias("tx_amount"),
    split_col.getItem(5).cast("long").alias("tx_time_seconds"),
    split_col.getItem(6).cast("long").alias("tx_time_days"),
    split_col.getItem(7).cast("int").alias("tx_fraud"),
    split_col.getItem(8).cast("int").alias("tx_fraud_scenario"),
)

df.printSchema()
df.show(5, truncate=False)

root
 |-- tranaction_id: long (nullable = true)
 |-- tx_datetime: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- terminal_id: long (nullable = true)
 |-- tx_amount: double (nullable = true)
 |-- tx_time_seconds: long (nullable = true)
 |-- tx_time_days: long (nullable = true)
 |-- tx_fraud: integer (nullable = true)
 |-- tx_fraud_scenario: integer (nullable = true)

+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|tranaction_id|tx_datetime        |customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|0            |2019-08-22 06:51:03|0          |711        |70.91    |24663          |0           |0       |0                |
|1            |2019-08-22 05:10:37|0          |0          |90.55    |18637          |0           |0    

In [23]:
print(f"Строк: {df.count():,}, колонок: {len(df.columns)}")

Строк: 46,988,418, колонок: 9


In [25]:
print("Пропуски:")
for col in df.columns:
    nulls_count = df.filter(F.col(col).isNull()).count()
    print(f"{col}: {nulls_count}")

Пропуски:
tranaction_id: 0
tx_datetime: 0
customer_id: 0
terminal_id: 0
tx_amount: 0
tx_time_seconds: 0
tx_time_days: 0
tx_fraud: 0
tx_fraud_scenario: 0


In [32]:
print(f"Число дубликатов: {df.count() - df.distinct().count()}")

Число дубликатов: 181


In [36]:
df.select("tx_amount").describe().show()

+-------+-----------------+
|summary|        tx_amount|
+-------+-----------------+
|  count|         46988418|
|   mean|54.23395999456729|
| stddev|41.25033514383644|
|    min|              0.0|
|    max|          3773.34|
+-------+-----------------+



In [39]:
print(f"tx_amount' неположительный: {df.filter(F.col('tx_amount') <= 0).count()}")

tx_amount' неположительный: 884


In [44]:
df.select("tx_time_days").describe().show()

+-------+-----------------+
|summary|     tx_time_days|
+-------+-----------------+
|  count|         46988418|
|   mean| 14.5006346883183|
| stddev|8.655415884394312|
|    min|                0|
|    max|               29|
+-------+-----------------+



In [45]:
df.select("tx_time_seconds").describe().show()

+-------+------------------+
|summary|   tx_time_seconds|
+-------+------------------+
|  count|          46988418|
|   mean|1296054.5003670265|
| stddev|  748049.508734945|
|    min|                 0|
|    max|           2592000|
+-------+------------------+



In [42]:
print(f"Некорректные даты: {df.withColumn('tx_ts', F.to_timestamp('tx_datetime')).filter(F.col('tx_ts').isNull()).count()}")

Некорректные даты: 100


In [43]:
df.select("tx_time_days").describe().show()

+-------+----------------+
|summary|    tx_time_days|
+-------+----------------+
|  count|        46988418|
|   mean|14.5006346883183|
| stddev|8.65541588439431|
|    min|               0|
|    max|              29|
+-------+----------------+



In [47]:
df.groupBy("tx_fraud").count().orderBy("tx_fraud").show()

+--------+--------+
|tx_fraud|   count|
+--------+--------+
|       0|44461413|
|       1| 2527005|
+--------+--------+



In [48]:
df.groupBy("tx_fraud_scenario").count().orderBy("tx_fraud_scenario").show()

+-----------------+--------+
|tx_fraud_scenario|   count|
+-----------------+--------+
|                0|44461413|
|                1|   25653|
|                2| 2435456|
|                3|   65896|
+-----------------+--------+

